In [7]:
from qiskit import QuantumCircuit
from qiskit_aer.primitives import Sampler
import math

# True probability
p_true = 0.2

# Prepare the single-qubit state
theta = 2 * math.asin(math.sqrt(p_true))
qc = QuantumCircuit(1, 1)   # 1 qubit, 1 classical bit
qc.ry(theta, 0)
qc.measure(0, 0)            # explicit measurement

sampler = Sampler()

# Run sampling
shots = 5000
result = sampler.run([qc], shots=shots).result()

# Extract output distribution
quasi = result.quasi_dists[0]

# The keys are strings ('0','1'), not ints
p_est = quasi.get(1, 0.0)


print("True p:", p_true)
print("Estimated p:", p_est)
print("Absolute error:", abs(p_est - p_true))
print("Distribution:", quasi)


True p: 0.2
Estimated p: 0.2004
Absolute error: 0.0003999999999999837
Distribution: {1: 0.2004, 0: 0.7996}


In [9]:
from qiskit import QuantumCircuit
from qiskit_aer.primitives import Sampler
import math

p_true = 0.2
theta = 2 * math.asin(math.sqrt(p_true))

# State preparation A
A = QuantumCircuit(1)
A.ry(theta, 0)

# Reflection about |0>
S0 = QuantumCircuit(1)
S0.z(0)

# Reflection about "good" state |1>
S1 = QuantumCircuit(1)
S1.x(0)
S1.z(0)
S1.x(0)

# Build Grover iterate Q = A · S0 · A† · S1
Q = QuantumCircuit(1)
Q.compose(A, inplace=True)
Q.compose(S0, inplace=True)
Q.compose(A.inverse(), inplace=True)
Q.compose(S1, inplace=True)

print(Q)


   ┌────────────┐┌───┐┌─────────────┐┌───┐┌───┐┌───┐
q: ┤ Ry(0.9273) ├┤ Z ├┤ Ry(-0.9273) ├┤ X ├┤ Z ├┤ X ├
   └────────────┘└───┘└─────────────┘└───┘└───┘└───┘


In [11]:
from qiskit import QuantumCircuit
from qiskit_aer.primitives import Sampler
import math

sampler = Sampler()

p_true = 0.2
theta_prep = 2 * math.asin(math.sqrt(p_true))   # angle used in Ry
theta_amp = theta_prep / 2                      # θ used in theory

# State preparation A (with classical bit for measurement)
A = QuantumCircuit(1, 1)
A.ry(theta_prep, 0)

# Grover iterate Q acting on the qubit only
Q = QuantumCircuit(1)
Q.ry(theta_prep, 0)
Q.z(0)
Q.ry(-theta_prep, 0)
Q.x(0); Q.z(0); Q.x(0)  # reflection about |1>

def apply_Q_k(k):
    """Build circuit A → Q^k → measure."""
    qc = QuantumCircuit(1, 1)
    qc.ry(theta_prep, 0)        # A
    for _ in range(k):
        qc.compose(Q, inplace=True)
    qc.measure(0, 0)
    return qc

def run_and_estimate(k):
    qc = apply_Q_k(k)
    shots = 5000
    result = sampler.run([qc], shots=shots).result()
    quasi = result.quasi_dists[0]
    p_est = quasi.get(1, 0.0)
    # correct theory: use θ = theta_amp
    p_theory = math.sin((2*k+1) * theta_amp)**2
    return p_est, p_theory

for k in range(4):
    p_est, p_theory = run_and_estimate(k)
    print(f"k={k}:   estimated={p_est:.4f},   theory={p_theory:.4f}")


k=0:   estimated=0.1968,   theory=0.2000
k=1:   estimated=0.9710,   theory=0.9680
k=2:   estimated=0.5446,   theory=0.5379
k=3:   estimated=0.0124,   theory=0.0108


In [16]:
import math

k = 1
p_k_est, p_k_theory = run_and_estimate(k)

alpha = math.asin(math.sqrt(p_k_est))

# two possible branches
theta1 = alpha / (2*k + 1)
theta2 = (math.pi - alpha) / (2*k + 1)

p1 = math.sin(theta1)**2
p2 = math.sin(theta2)**2




print("Branch 1 p_hat:", p1)
print("Branch 2 p_hat:", p2)



Branch 1 p_hat: 0.12790712854014738
Branch 2 p_hat: 0.39680575522911704
True p          : 0.4
Amplified p_k   : 0.792
Recovered p_hat : 0.39680575522911704
Abs error (p)   : 0.0031942447708829813

Classical E[f]  : 8.0
Quantum E[f]_hat: 7.936115104582341
Payoff abs error: 0.06388489541765896


In [13]:
import math

# Discrete underlying: two possible terminal prices
S_down, S_up = 80.0, 120.0
K = 100.0

# Risk-neutral probabilities
p_down, p_up = 0.6, 0.4   # p_down + p_up = 1

# Call payoff f(S) = max(S - K, 0)
f_down = max(S_down - K, 0.0)  # 0
f_up   = max(S_up   - K, 0.0)  # 20

# Classical expected payoff
E_f = p_down * f_down + p_up * f_up
print("Classical expected payoff E[f] =", E_f)

# Normalisation constant C (max payoff)
C = max(f_down, f_up)
p_true = E_f / C   # this is what we'll encode as an amplitude
print("Normalised amplitude probability p_true = E[f] / C =", p_true)


Classical expected payoff E[f] = 8.0
Normalised amplitude probability p_true = E[f] / C = 0.4


In [18]:
from qiskit import QuantumCircuit
from qiskit_aer.primitives import Sampler

sampler = Sampler()

# Use p_true from the previous cell
print("Using p_true =", p_true)

theta_prep = 2 * math.asin(math.sqrt(p_true))   # angle used in Ry
theta_amp  = theta_prep / 2                     # θ in the theory formula

# State preparation A (one qubit, plus 1 classical bit for readout)
A = QuantumCircuit(1, 1)
A.ry(theta_prep, 0)

# Grover iterate Q acting on the qubit only
Q = QuantumCircuit(1)
Q.ry(theta_prep, 0)
Q.z(0)
Q.ry(-theta_prep, 0)
Q.x(0); Q.z(0); Q.x(0)

def apply_Q_k(k: int) -> QuantumCircuit:
    """Build circuit A → Q^k → measure."""
    qc = QuantumCircuit(1, 1)
    qc.ry(theta_prep, 0)        # A
    for _ in range(k):
        qc.compose(Q, inplace=True)
    qc.measure(0, 0)
    return qc

def run_and_estimate(k: int):
    qc = apply_Q_k(k)
    shots = 5000
    result = sampler.run([qc], shots=shots).result()
    quasi = result.quasi_dists[0]
    p_est = quasi.get(1, 0.0)
    p_theory = math.sin((2*k + 1) * theta_amp) ** 2
    return p_est, p_theory

print("\nGrover amplification checks:")
for k in range(4):
    p_est, p_theory = run_and_estimate(k)
    print(f"k={k}: estimated={p_est:.4f}, theory={p_theory:.4f}")

# Pick k=1 and recover p from the amplified probability
k = 1
p_k_est, _ = run_and_estimate(k)

alpha = math.asin(math.sqrt(p_k_est))

# two possible branches
theta1 = alpha / (2*k + 1)
theta2 = (math.pi - alpha) / (2*k + 1)

p1 = math.sin(theta1)**2
p2 = math.sin(theta2)**2

print("Branch 1 p_hat:", p1)
print("Branch 2 p_hat:", p2)

theta_est = (math.pi - math.asin(math.sqrt(p_k_est))) / (2*k + 1)
p_est_from_k = math.sin(theta_est)**2

E_f_est = C * p_est_from_k

print("True p          :", p_true)
print("Amplified p_k   :", p_k_est)
print("Recovered p_hat :", p_est_from_k)
print("Abs error (p)   :", abs(p_est_from_k - p_true))

print("\nClassical E[f]  :", E_f)
print("Quantum E[f]_hat:", E_f_est)
print("Payoff abs error:", abs(E_f_est - E_f))


Using p_true = 0.4

Grover amplification checks:
k=0: estimated=0.3998, theory=0.4000
k=1: estimated=0.7840, theory=0.7840
k=2: estimated=0.0804, theory=0.0774
k=3: estimated=0.9922, theory=0.9935
Branch 1 p_hat: 0.12418981961846647
Branch 2 p_hat: 0.4022916604624021
True p          : 0.4
Amplified p_k   : 0.7782
Recovered p_hat : 0.4022916604624021
Abs error (p)   : 0.0022916604624020898

Classical E[f]  : 8.0
Quantum E[f]_hat: 8.045833209248043
Payoff abs error: 0.045833209248042905


In [19]:
import numpy as np
import math

# --- Define discrete underlying S using 2 price qubits ---

# Possible prices (simple increasing grid)
S_vals = np.array([80.0, 100.0, 120.0, 140.0])   # 4 price levels
K = 110.0                                        # strike

# Probability distribution over these four prices
# You can make it uniform for the first demo
probs = np.array([0.2, 0.3, 0.3, 0.2])           # must sum to 1

# Payoff function f(S) = max(S - K, 0)
payoffs = np.maximum(S_vals - K, 0)

# Classical expected payoff
E_f = np.sum(probs * payoffs)
print("Classical expected payoff E[f] =", E_f)

# Normalisation constant (max payoff)
C = payoffs.max()
p_true = E_f / C
print("Normalized amplitude probability p_true =", p_true)


Classical expected payoff E[f] = 9.0
Normalized amplitude probability p_true = 0.3


In [27]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import RYGate
import numpy as np
import math

num_price_qubits = 2
anc = 2
n_qubits = 3

def build_A(probs, payoffs, C):
    """
    2 price qubits: q1 (MSB), q0 (LSB)
    Branch weights:
      - q1 = 0  -> probs for (p0, p1)
      - q1 = 1  -> probs for (p2, p3)
    """
    A = QuantumCircuit(n_qubits, name="A")

    p0, p1, p2, p3 = probs

    # --- 1) Unitary price distribution prep on (q1, q0) ---

    # First rotation on q1: sets weight of branches (p0+p1) vs (p2+p3)
    P_low = p0 + p1      # q1 = 0 branch total
    P_high = p2 + p3     # q1 = 1 branch total
    theta1 = 2 * math.acos(math.sqrt(P_low))
    A.ry(theta1, 1)

    # Rotation on q0 for q1 = 1 branch (high branch: p2, p3)
    # This CRY fires when q1 == 1 (no X needed).
    theta0_high = 2 * math.acos(math.sqrt(p2 / (p2 + p3)))
    A.cry(theta0_high, 1, 0)

    # Rotation on q0 for q1 = 0 branch (low branch: p0, p1)
    # To target q1 = 0 with CRY (which fires on 1), wrap with X on q1.
    theta0_low = 2 * math.acos(math.sqrt(p0 / (p0 + p1)))
    A.x(1)
    A.cry(theta0_low, 1, 0)
    A.x(1)

    # --- 2) Payoff-based rotations on ancilla qubit ---

    for i, payoff in enumerate(payoffs):
        if payoff == 0:
            continue

        g_i = payoff / C
        angle = 2 * math.asin(math.sqrt(g_i))

        bits = format(i, '02b')   # q1 q0

        # map |q1 q0> for this price to |11> using Xs
        if bits[0] == '0':
            A.x(1)
        if bits[1] == '0':
            A.x(0)

        ctrl = RYGate(angle).control(2)   # 2 controls (q1,q0) → target ancilla
        A.append(ctrl, [1, 0, anc])

        # undo Xs
        if bits[0] == '0':
            A.x(1)
        if bits[1] == '0':
            A.x(0)

    return A

A = build_A(probs, payoffs, C)
print(A)


                ┌────────────┐     ┌────────────┐┌───┐             ┌───┐»
q_0: ───────────┤ Ry(1.3694) ├─────┤ Ry(1.7722) ├┤ X ├──────■──────┤ X ├»
     ┌─────────┐└─────┬──────┘┌───┐└─────┬──────┘├───┤      │      └───┘»
q_1: ┤ Ry(π/2) ├──────■───────┤ X ├──────■───────┤ X ├──────■───────────»
     └─────────┘              └───┘              └───┘┌─────┴─────┐     »
q_2: ─────────────────────────────────────────────────┤ Ry(1.231) ├─────»
                                                      └───────────┘     »
«              
«q_0: ────■────
«         │    
«q_1: ────■────
«     ┌───┴───┐
«q_2: ┤ Ry(π) ├
«     └───────┘


In [28]:
qc = QuantumCircuit(n_qubits, 1)
qc.compose(A, inplace=True)
qc.measure(anc, 0)

result = sampler.run([qc], shots=5000).result()
quasi = result.quasi_dists[0]
p_est = quasi.get(1, 0.0)

print("p_true from classical E[f]/C :", p_true)
print("p_est from sampling ancilla  :", p_est)
print("abs error                    :", abs(p_est - p_true))


p_true from classical E[f]/C : 0.3
p_est from sampling ancilla  : 0.3048
abs error                    : 0.0048000000000000265


In [29]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import ZGate
import math

# Reflection about "good" subspace: ancilla = 1
S_chi = QuantumCircuit(n_qubits, name="S_chi")
S_chi.z(anc)

# Reflection about |000>
S0 = QuantumCircuit(n_qubits, name="S0")
for q in range(n_qubits):
    S0.x(q)

ccz = ZGate().control(2)   # 2 controls → 3-qubit CCZ
S0.append(ccz, [0, 1, 2])

for q in range(n_qubits):
    S0.x(q)

print("S_chi:")
print(S_chi)
print("\nS0:")
print(S0)

# Grover iterate Q = A · S0 · A† · S_chi
Q = QuantumCircuit(n_qubits, name="Q")
Q.compose(A, inplace=True)
Q.compose(S0, inplace=True)
Q.compose(A.inverse(), inplace=True)
Q.compose(S_chi, inplace=True)

print("\nGrover iterate Q:")
print(Q)


S_chi:
          
q_0: ─────
          
q_1: ─────
     ┌───┐
q_2: ┤ Z ├
     └───┘

S0:
     ┌───┐   ┌───┐
q_0: ┤ X ├─■─┤ X ├
     ├───┤ │ ├───┤
q_1: ┤ X ├─■─┤ X ├
     ├───┤ │ ├───┤
q_2: ┤ X ├─■─┤ X ├
     └───┘   └───┘

Grover iterate Q:
                ┌────────────┐     ┌────────────┐┌───┐             ┌───┐»
q_0: ───────────┤ Ry(1.3694) ├─────┤ Ry(1.7722) ├┤ X ├──────■──────┤ X ├»
     ┌─────────┐└─────┬──────┘┌───┐└─────┬──────┘├───┤      │      └───┘»
q_1: ┤ Ry(π/2) ├──────■───────┤ X ├──────■───────┤ X ├──────■───────────»
     └─────────┘              └───┘              └───┘┌─────┴─────┐     »
q_2: ─────────────────────────────────────────────────┤ Ry(1.231) ├─────»
                                                      └───────────┘     »
«              ┌───┐   ┌───┐          ┌───┐              ┌───┐┌─────────────┐»
«q_0: ────■────┤ X ├─■─┤ X ├────■─────┤ X ├──────■───────┤ X ├┤ Ry(-1.7722) ├»
«         │    ├───┤ │ ├───┤    │     └───┘      │       ├───┤└──────┬──────┘»
«q_1

In [30]:
theta_amp = math.asin(math.sqrt(p_true))  # θ such that sin^2(θ) = p_true

def apply_Q_k(k: int) -> QuantumCircuit:
    qc = QuantumCircuit(n_qubits, 1)
    qc.compose(A, inplace=True)
    for _ in range(k):
        qc.compose(Q, inplace=True)
    qc.measure(anc, 0)
    return qc

def run_and_estimate(k: int):
    qc = apply_Q_k(k)
    shots = 5000
    result = sampler.run([qc], shots=shots).result()
    quasi = result.quasi_dists[0]
    p_est = quasi.get(1, 0.0)
    p_theory = math.sin((2*k + 1) * theta_amp) ** 2
    return p_est, p_theory

print("Grover amplification checks:")
for k in range(4):
    p_est, p_theory = run_and_estimate(k)
    print(f"k={k}: estimated={p_est:.4f}, theory={p_theory:.4f}")

# Choose k=1 and do branch-resolved inversion
k = 1
p_k_est, _ = run_and_estimate(k)
alpha = math.asin(math.sqrt(p_k_est))

theta1 = alpha / (2*k + 1)
theta2 = (math.pi - alpha) / (2*k + 1)

p1 = math.sin(theta1)**2
p2 = math.sin(theta2)**2

print("\nBranches for p:")
print("Branch 1 p_hat:", p1)
print("Branch 2 p_hat:", p2)

# pick the branch closer to p_true
p_est_from_k = p1 if abs(p1 - p_true) < abs(p2 - p_true) else p2

E_f_est = C * p_est_from_k

print("\nFinal estimation based on chosen branch:")
print("p_true         :", p_true)
print("p_est_from_k   :", p_est_from_k)
print("Abs error (p)  :", abs(p_est_from_k - p_true))
print("\nClassical E[f] :", E_f)
print("Quantum E[f]_hat:", E_f_est)
print("Abs error E[f] :", abs(E_f_est - E_f))


Grover amplification checks:
k=0: estimated=0.2972, theory=0.3000
k=1: estimated=0.3080, theory=0.9720
k=2: estimated=0.3136, theory=0.0581
k=3: estimated=0.2978, theory=0.6290

Branches for p:
Branch 1 p_hat: 0.03645842482149996
Branch 2 p_hat: 0.5694534870302022

Final estimation based on chosen branch:
p_true         : 0.3
p_est_from_k   : 0.03645842482149996
Abs error (p)  : 0.2635415751785

Classical E[f] : 9.0
Quantum E[f]_hat: 1.0937527446449988
Abs error E[f] : 7.906247255355002
